In [1]:
# GPU benchmark setup — 2026-10-15
import sys, subprocess, os
from pathlib import Path
repo = Path('/content/hw2_speed_check')
if not repo.exists():
    subprocess.run(['git','clone','--branch','2026','https://github.com/OsVox/hse-nlp.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.53.0','pyarrow','requests'],check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime'
print('GPU:',torch.cuda.get_device_name())
print('torch:',torch.__version__)
print('revision:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
torch: 2.11.0+cu130
revision: 93f1a5f474a06800a33d67ae412eefa235a84c11


In [2]:
import csv, gc, hashlib, json, subprocess, types, time, statistics
from pathlib import Path
import torch, transformers
from homeworks.hw2.inference import generate_predictions
from homeworks.hw2.data import fetch_parquet

ROOT=Path('/content/hw2_speed_check')
OUT=Path('/content/hw2_gpu_results');OUT.mkdir(exist_ok=True)
CACHE=Path('/content/hw2_speed_cache')
MODEL=ROOT/'homeworks/hw2/model'
old=types.ModuleType('homeworks.hw2.before');old.__package__='homeworks.hw2'
exec(subprocess.check_output(['git','show','3c5cb90:homeworks/hw2/inference.py'],text=True),old.__dict__)
fetch_parquet('data/test/test.parquet',CACHE)
# Прогрев CUDA и модели до измерений. Веса и параметры генерации одинаковы.
tok=transformers.PreTrainedTokenizerFast.from_pretrained(MODEL)
for dtype in [torch.float32,torch.float16,torch.bfloat16]:
 m=transformers.T5ForConditionalGeneration.from_pretrained(MODEL,torch_dtype=dtype).cuda().eval()
 with torch.inference_mode():
  m.generate(**tok(['calculate the sum of two numbers'],return_tensors='pt',return_token_type_ids=False).to('cuda'),max_new_tokens=16)
 del m;gc.collect();torch.cuda.empty_cache()
variants={'before':(old.generate_predictions,{}),'after_fp32':(generate_predictions,{'precision':'fp32'}),'after_fp16':(generate_predictions,{'precision':'fp16'}),'after_bf16':(generate_predictions,{'precision':'bf16'})}
records=[];outputs={}
for rep, names in enumerate([list(variants),list(reversed(variants))]):
 for name in names:
  fn,kw=variants[name];path=OUT/f'{name}_{rep}.csv'
  count,seconds=fn(MODEL,path,CACHE,batch_size=64,max_new_tokens=192,**kw)
  with path.open(newline='') as f: answers=[r['code'] for r in csv.DictReader(f)]
  target=OUT/f'{name}.jsonl'
  target.write_text(''.join(json.dumps({'code':s},ensure_ascii=False)+'\n' for s in answers))
  records.append({'variant':name,'repeat':rep,'seconds':seconds,'rows':count,'sha256':hashlib.sha256(target.read_bytes()).hexdigest()})
  if name in outputs: assert answers==outputs[name],f'Non-deterministic {name}'
  outputs[name]=answers
  print('RUN',json.dumps(records[-1]),flush=True)
  del answers;gc.collect();torch.cuda.empty_cache()
summary={'gpu':torch.cuda.get_device_name(),'torch':torch.__version__,'transformers':transformers.__version__,'before_commit':'3c5cb90','after_commit':'93f1a5f','batch_size':64,'max_new_tokens':192,'model_sha256':hashlib.sha256((MODEL/'model.safetensors').read_bytes()).hexdigest(),'runs':records,'median_seconds':{n:statistics.median(r['seconds'] for r in records if r['variant']==n) for n in variants},'changed_answers':{n:sum(a!=b for a,b in zip(outputs['before'],outputs[n])) for n in variants}}
(OUT/'inference_results.json').write_text(json.dumps(summary,indent=2))
print('SUMMARY',json.dumps(summary),flush=True)


generated=1919 seconds=11.40 device=cuda invalid_before_retry=8 beam_repaired=7 syntax_repaired=1
RUN {"variant": "before", "repeat": 0, "seconds": 11.39617088400064, "rows": 1919, "sha256": "661929273d93744c73ea64033fb6057722a9116cc8c9b5676ac0bf13db1e0ee6"}


You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


generated=1919 seconds=11.12 device=cuda precision=torch.float32 invalid_before_retry=8 beam_repaired=7 syntax_repaired=1
RUN {"variant": "after_fp32", "repeat": 0, "seconds": 11.117034443000193, "rows": 1919, "sha256": "661929273d93744c73ea64033fb6057722a9116cc8c9b5676ac0bf13db1e0ee6"}


You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


generated=1919 seconds=12.97 device=cuda precision=torch.float16 invalid_before_retry=7 beam_repaired=6 syntax_repaired=1
RUN {"variant": "after_fp16", "repeat": 0, "seconds": 12.970682344000124, "rows": 1919, "sha256": "a92b8ac160943969a0a8070a2759e33ba13e1ee46ba81cf7819e3ad2640f2278"}


...

generated=1919 seconds=12.22 device=cuda precision=torch.bfloat16 invalid_before_retry=11 beam_repaired=10 syntax_repaired=1
RUN {"variant": "after_bf16", "repeat": 1, "seconds": 12.21895614999994, "rows": 1919, "sha256": "7b809aa516ce41d534c0fc8a8666d8533f2765f0b401d37c5326193b97ad6200"}


You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


generated=1919 seconds=13.08 device=cuda precision=torch.float16 invalid_before_retry=7 beam_repaired=6 syntax_repaired=1
RUN {"variant": "after_fp16", "repeat": 1, "seconds": 13.08404956600043, "rows": 1919, "sha256": "a92b8ac160943969a0a8070a2759e33ba13e1ee46ba81cf7819e3ad2640f2278"}


You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


generated=1919 seconds=11.36 device=cuda precision=torch.float32 invalid_before_retry=8 beam_repaired=7 syntax_repaired=1
RUN {"variant": "after_fp32", "repeat": 1, "seconds": 11.363347521999458, "rows": 1919, "sha256": "661929273d93744c73ea64033fb6057722a9116cc8c9b5676ac0bf13db1e0ee6"}
generated=1919 seconds=11.30 device=cuda invalid_before_retry=8 beam_repaired=7 syntax_repaired=1
RUN {"variant": "before", "repeat": 1, "seconds": 11.303473787000257, "rows": 1919, "sha256": "661929273d93744c73ea64033fb6057722a9116cc8c9b5676ac0bf13db1e0ee6"}
SUMMARY {"gpu": "NVIDIA RTX PRO 6000 Blackwell Server Edition", "torch": "2.11.0+cu130", "transformers": "4.53.0", "before_commit": "3c5cb90", "after_commit": "93f1a5f", "batch_size": 64, "max_new_tokens": 192, "model_sha256": "987a9f2b0837a718430704887d1c05bda05c0479d86d8570df14d8275b7fbaa0", "runs": [{"variant": "before", "repeat": 0, "seconds": 11.39617088400064, "rows": 1919, "sha256": "661929273d93744c73ea64033fb6057722a9116cc8c9b5676ac0bf13db

In [3]:
import contextlib, copy, gc, json, time
from pathlib import Path
import pyarrow.parquet as pq
import torch
from transformers import PreTrainedTokenizerFast, T5ForConditionalGeneration
from homeworks.hw2.data import training_shard, VALIDATION_ROWS
from homeworks.hw2.train import encode_batch, validation_loss
from homeworks.hw2.speed import group_batches

out=Path('/content/hw2_gpu_results')
path=training_shard(0,Path('/content/hw2_speed_cache'))
rows=pq.read_table(path,columns=['question','code']).slice(0,VALIDATION_ROWS+5120).to_pylist()
validation=rows[256:512]
train_rows=rows[VALIDATION_ROWS:VALIDATION_ROWS+5120]
assert len(train_rows)==5120
import random
random.Random(42).shuffle(train_rows)
tok=PreTrainedTokenizerFast.from_pretrained('homeworks/hw2/model')
results=[]
for name,dtype,window,cache,sync in [('before',torch.float16,1,True,True),('after_fp16',torch.float16,32,False,False),('after_bf16',torch.bfloat16,32,False,False)]:
 torch.manual_seed(42)
 model=T5ForConditionalGeneration.from_pretrained('homeworks/hw2/model').cuda().train()
 optimizer=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=0.01)
 scaler=torch.amp.GradScaler('cuda',enabled=dtype==torch.float16)
 batches=iter([train_rows[i:i+16] for i in range(0,len(train_rows),16)])
 if window>1:batches=group_batches(batches,tok,16,window)
 times=[]
 for step in range(80):
  torch.cuda.synchronize();start=time.monotonic()
  optimizer.zero_grad(set_to_none=True)
  loss_sum=0.0 if sync else torch.zeros((),device='cuda')
  for _ in range(4):
   batch=encode_batch(next(batches),tok,torch.device('cuda'))
   with torch.autocast('cuda',dtype=dtype):loss=model(**batch,use_cache=cache).loss/4
   scaler.scale(loss).backward()
   loss_sum+=float(loss.detach()) if sync else loss.detach()
  scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
  scaler.step(optimizer);scaler.update()
  torch.cuda.synchronize();times.append(time.monotonic()-start)
 score=validation_loss(model,validation,tok,torch.device('cuda'),16)
 record={'variant':name,'updates':80,'warmup_updates':8,'timed_seconds':sum(times[8:]),'updates_per_second':72/sum(times[8:]),'validation_token_loss':score,'last_train_loss':float(loss_sum)}
 results.append(record);print('TRAIN_PROFILE',json.dumps(record),flush=True)
 del model,optimizer,scaler;gc.collect();torch.cuda.empty_cache()
(out/'training_results.json').write_text(json.dumps({'gpu':torch.cuda.get_device_name(),'runs':results,'note':'Short 80-update continuation from identical checkpoint on identical 5120 rows; grouping changes order. Not a full training quality guarantee; weights discarded.'},indent=2))


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


TRAIN_PROFILE {"variant": "before", "updates": 80, "warmup_updates": 8, "timed_seconds": 5.72433942099542, "updates_per_second": 12.577870511298181, "validation_token_loss": 0.9151512504539332, "last_train_loss": 1.1234943866729736}
TRAIN_PROFILE {"variant": "after_fp16", "updates": 80, "warmup_updates": 8, "timed_seconds": 5.779089166999256, "updates_per_second": 12.458710692879896, "validation_token_loss": 0.9152673313833171, "last_train_loss": 1.4157063961029053}
TRAIN_PROFILE {"variant": "after_bf16", "updates": 80, "warmup_updates": 8, "timed_seconds": 5.713917413001582, "updates_per_second": 12.600812156677224, "validation_token_loss": 0.9152718785417481, "last_train_loss": 1.4154893159866333}


1075